In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from datetime import datetime
from matplotlib.backends.backend_pdf import PdfPages

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
data_path = r"C:\Users\adity\Theta A\DA Project\cleaned_data"

users = pd.read_csv(os.path.join(data_path, "users_clean.csv"))
drivers = pd.read_csv(os.path.join(data_path, "drivers_clean.csv"))
vehicles = pd.read_csv(os.path.join(data_path, "vehicles_clean.csv"))
rides = pd.read_csv(os.path.join(data_path, "rides_clean.csv"))
ratings = pd.read_csv(os.path.join(data_path, "ratings_clean.csv"))

print("All datasets loaded successfully!")

print("Users:", users.shape)
print("Drivers:", drivers.shape)
print("Vehicles:", vehicles.shape)
print("Rides:", rides.shape)
print("Ratings:", ratings.shape)

All datasets loaded successfully!
Users: (10000, 8)
Drivers: (300, 6)
Vehicles: (300, 5)
Rides: (50000, 9)
Ratings: (50000, 6)


In [3]:
# Convert date columns
rides["ride_start_time"] = pd.to_datetime(rides["ride_start_time"])
rides["ride_end_time"] = pd.to_datetime(rides["ride_end_time"])
ratings["rating_date"] = pd.to_datetime(ratings["rating_date"])

# Create additional columns
rides["ride_duration_min"] = (
    rides["ride_end_time"] - rides["ride_start_time"]
).dt.total_seconds() / 60

rides["fare_per_km"] = rides["fare_amount"] / rides["distance_km"]

rides["ride_month"] = rides["ride_start_time"].dt.to_period("M").astype(str)
rides["ride_hour"] = rides["ride_start_time"].dt.hour

# Generate statistical summary
statistical_summary = rides[
    ["distance_km", "fare_amount", "ride_duration_min", "fare_per_km"]
].describe().round(2)

display(statistical_summary)

,distance_km,fare_amount,ride_duration_min,fare_per_km
count,50000.00,50000.00,50000.00,50000.00
mean,25.54,57.43,62.46,2.25
std,14.16,34.26,33.42,0.43
min,1.00,1.57,5.00,1.50
25%,13.22,28.67,33.00,1.88
50%,25.54,55.29,62.00,2.25
75%,37.89,82.01,91.00,2.62
max,50.00,149.13,120.00,3.00


In [4]:
total_rides = len(rides)
total_revenue = rides["fare_amount"].sum()
average_fare = rides["fare_amount"].mean()
average_distance = rides["distance_km"].mean()
average_duration = rides["ride_duration_min"].mean()
average_customer_rating = ratings["rating_value"].mean()

total_users = users["user_id"].nunique()
total_drivers = drivers["driver_id"].nunique()
total_vehicles = vehicles["vehicle_id"].nunique()

kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Rides",
        "Total Revenue",
        "Average Fare",
        "Average Distance",
        "Average Ride Duration",
        "Average Customer Rating",
        "Total Users",
        "Total Drivers",
        "Total Vehicles"
    ],
    "Value": [
        total_rides,
        round(total_revenue, 2),
        round(average_fare, 2),
        round(average_distance, 2),
        round(average_duration, 2),
        round(average_customer_rating, 2),
        total_users,
        total_drivers,
        total_vehicles
    ]
})

display(kpi_summary)

,Metric,Value
0,Total Rides,50000.00
1,Total Revenue,2871581.19
2,Average Fare,57.43
3,Average Distance,25.54
4,Average Ride Duration,62.46
5,Average Customer Rating,3.01
6,Total Users,10000.00
7,Total Drivers,300.00
8,Total Vehicles,300.00


In [5]:
monthly_bookings = (
    rides.groupby("ride_month")
    .agg(
        Total_Rides=("ride_id", "count"),
        Total_Revenue=("fare_amount", "sum")
    )
    .reset_index()
)

monthly_bookings["Total_Revenue"] = monthly_bookings["Total_Revenue"].round(2)

display(monthly_bookings)

,ride_month,Total_Rides,Total_Revenue
0,2024-01,5643,323370.74
1,2024-02,5234,303440.31
2,2024-03,5622,321832.35
3,2024-04,5198,302229.25
4,2024-05,5654,319794.91
5,2024-06,5476,320555.98
6,2024-07,5640,320353.97
7,2024-08,5505,312184.14
8,2024-09,5410,312138.84
9,2024-10,618,35680.70


In [6]:
hourly_bookings = (
    rides.groupby("ride_hour")
    .agg(Total_Rides=("ride_id", "count"))
    .reset_index()
    .sort_values("Total_Rides", ascending=False)
)

display(hourly_bookings)

,ride_hour,Total_Rides
19,19,2161
6,6,2137
0,0,2131
13,13,2122
22,22,2114
23,23,2112
1,1,2112
7,7,2111
9,9,2109
17,17,2101


In [7]:
driver_performance = (
    rides.groupby("driver_id")
    .agg(
        Completed_Rides=("ride_id", "count"),
        Total_Revenue=("fare_amount", "sum")
    )
    .reset_index()
)

driver_performance = driver_performance.merge(
    drivers[["driver_id", "name", "rating"]],
    on="driver_id",
    how="left"
)

driver_performance = driver_performance.sort_values(
    "Completed_Rides", ascending=False
).head(10)

driver_performance["Total_Revenue"] = driver_performance["Total_Revenue"].round(2)

display(driver_performance)

,driver_id,Completed_Rides,Total_Revenue,name,rating
89,90,201,11510.15,Jeremy Ramirez,2.30
291,292,197,12480.69,Lisa Rojas,1.63
170,171,195,11028.12,Aaron Crawford,2.28
148,149,194,10880.88,John Foster,2.12
196,197,194,10720.26,Jeremy Bautista,2.24
14,15,194,10869.99,Amber Taylor,2.45
147,148,193,11216.92,Christopher Miller,1.88
127,128,191,11041.28,Alicia Moore,4.71
282,283,191,11095.25,Holly Herrera,2.03
23,24,191,10214.81,John Hill,2.91


In [8]:
top_pickup_locations = (
    rides["start_location"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_pickup_locations.columns = ["Pickup_Location", "Total_Rides"]

display(top_pickup_locations)

,Pickup_Location,Total_Rides
0,South Michael,50
1,Port Michael,48
2,New Michael,46
3,East Michael,44
4,West Michael,42
5,Lake Michael,41
6,East Jennifer,37
7,North Michael,36
8,South Robert,34
9,West Christopher,33


In [9]:
rating_distribution = (
    ratings["rating_value"]
    .value_counts()
    .sort_index()
    .reset_index()
)

rating_distribution.columns = ["Rating", "Total_Ratings"]

display(rating_distribution)

,Rating,Total_Ratings
0,1,9888
1,2,9928
2,3,10206
3,4,9861
4,5,10117


In [10]:
# Create a folder for report charts
chart_folder = "report_charts"
os.makedirs(chart_folder, exist_ok=True)

print("Chart folder created successfully!")

Chart folder created successfully!


In [13]:
chart_files = os.listdir(chart_folder)

print("Total exported charts:", len(chart_files))

for file in chart_files:
    print(file)

Total exported charts: 10
correlation_heatmap.png
customer_rating_distribution.png
distance_vs_fare.png.png
driver_rating_distribution.png.png
fare_and_distance_distribution.png
monthly_booking_trend.png
pair_plot.png
top_10_drivers.png
top_10_drivers.png.png
Top_10_Pickup_Locations.png


In [14]:
os.rename(
    os.path.join(chart_folder, "distance_vs_fare.png.png"),
    os.path.join(chart_folder, "distance_vs_fare.png")
)

os.rename(
    os.path.join(chart_folder, "driver_rating_distribution.png.png"),
    os.path.join(chart_folder, "driver_rating_distribution.png")
)

os.remove(
    os.path.join(chart_folder, "top_10_drivers.png.png")
)

print("Chart filenames corrected successfully!")

Chart filenames corrected successfully!


In [15]:
chart_files = sorted(os.listdir(chart_folder))

print("Total charts:", len(chart_files))

for file in chart_files:
    print(file)

Total charts: 9
Top_10_Pickup_Locations.png
correlation_heatmap.png
customer_rating_distribution.png
distance_vs_fare.png
driver_rating_distribution.png
fare_and_distance_distribution.png
monthly_booking_trend.png
pair_plot.png
top_10_drivers.png


In [16]:
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import pyplot as plt

report_path = os.path.join(
    os.getcwd(),
    "Ride_Sharing_Project_Report.pdf"
)

print("PDF report path:", report_path)

PDF report path: C:\Users\adity\Theta A\DA Project\Ride_Sharing_Project_Report.pdf


In [17]:
with PdfPages(report_path) as pdf:

    fig = plt.figure(figsize=(8.27, 11.69))

    plt.axis("off")

    plt.text(
        0.5, 0.75,
        "RIDE SHARING & CAB BOOKING ANALYSIS",
        fontsize=20,
        fontweight="bold",
        ha="center"
    )

    plt.text(
        0.5, 0.62,
        "Data Analysis Project Report",
        fontsize=16,
        ha="center"
    )

    plt.text(
        0.5, 0.48,
        "Analysis Period: January 2024 – October 2024",
        fontsize=12,
        ha="center"
    )

    plt.text(
        0.5, 0.40,
        "Prepared using Python, Pandas, SQL and Power BI",
        fontsize=11,
        ha="center"
    )

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print("PDF title page created successfully!")

PDF title page created successfully!


In [18]:
pdf = PdfPages(report_path)

print("PDF is ready for adding report sections!")

PDF is ready for adding report sections!


In [19]:
fig = plt.figure(figsize=(8.27, 11.69))
plt.axis("off")

plt.text(
    0.5, 0.75,
    "RIDE SHARING & CAB BOOKING ANALYSIS",
    fontsize=20,
    fontweight="bold",
    ha="center"
)

plt.text(
    0.5, 0.62,
    "Data Analysis Project Report",
    fontsize=16,
    ha="center"
)

plt.text(
    0.5, 0.48,
    "Analysis Period: January 2024 – October 2024",
    fontsize=12,
    ha="center"
)

plt.text(
    0.5, 0.40,
    "Prepared using Python, Pandas, SQL and Power BI",
    fontsize=11,
    ha="center"
)

pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

print("Title page added to PDF!")

Title page added to PDF!


In [20]:
fig = plt.figure(figsize=(8.27, 11.69))
plt.axis("off")

plt.text(0.08, 0.94, "1. PROJECT OVERVIEW",
         fontsize=17, fontweight="bold")

overview = """
Project Objective:

The objective of this project is to analyze ride-sharing
and cab-booking operations to understand booking trends,
revenue, driver performance, customer ratings and
operational patterns.

Key Areas of Analysis:

• Booking performance and demand patterns
• Revenue and fare analysis
• Driver performance and ratings
• Customer ratings
• Pickup-location analysis
• Trip distance and duration

Tools Used:

• Python
• Pandas and NumPy
• Matplotlib and Seaborn
• MySQL
• Power BI

2. DATASET INFORMATION

Dataset          Records       Columns
--------------------------------------
Users            10,000           8
Drivers             300           6
Vehicles            300           5
Rides            50,000           9
Ratings          50,000           6

Total Records: 110,600
Total Datasets: 5
"""

plt.text(0.08, 0.86, overview, fontsize=12, va="top",
         linespacing=1.6, family="monospace")

pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

print("Project overview and dataset information added!")

Project overview and dataset information added!


In [21]:
fig = plt.figure(figsize=(8.27, 11.69))
plt.axis("off")

plt.text(0.08, 0.94, "3. KEY PERFORMANCE INDICATORS (KPIs)",
         fontsize=16, fontweight="bold")

kpi_text = """
Overall Project Performance

Total Rides                  50,000
Total Revenue                2,871,581.19
Average Fare                 57.43
Average Trip Distance        25.54 km
Average Ride Duration        62.46 minutes
Average Customer Rating      3.01 / 5

Platform Overview

Total Users                  10,000
Total Drivers                300
Total Vehicles               300

Key Observations

• The dataset contains 50,000 ride records.
• Total recorded revenue is 2,871,581.19.
• Average trip distance is 25.54 km.
• Average ride duration is approximately 62 minutes.
• The average customer rating is 3.01 out of 5.

Note: The dataset does not specify the currency
for fare amounts.
"""

plt.text(0.10, 0.85, kpi_text, fontsize=13,
         va="top", linespacing=1.8, family="monospace")

pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

print("KPI summary added to PDF!")

KPI summary added to PDF!


In [22]:
fig = plt.figure(figsize=(8.27, 11.69))
plt.axis("off")

plt.text(0.08, 0.94, "4. STATISTICAL SUMMARY",
         fontsize=17, fontweight="bold")

stats_text = """
The following statistics describe the distribution
of key ride-related variables.

Metric              Distance    Fare       Duration    Fare/KM
----------------------------------------------------------------
Count               50,000      50,000     50,000      50,000
Mean                25.54       57.43      62.46       2.25
Standard Deviation  14.16       34.26      33.42       0.43
Minimum              1.00        1.57       5.00       1.50
25%                 13.22       28.67      33.00       1.88
Median              25.54       55.29      62.00       2.25
75%                 37.89       82.01      91.00       2.62
Maximum             50.00      149.13     120.00       3.00

Key Observations

• Average trip distance is 25.54 km.
• Average fare is 57.43.
• Average ride duration is 62.46 minutes.
• Fare per kilometre averages 2.25.
• The dataset contains rides with different distances,
  fares and durations, showing variation in ride patterns.

Note: Fare currency is not specified in the dataset.
"""

plt.text(0.06, 0.85, stats_text, fontsize=10.5,
         va="top", linespacing=1.8, family="monospace")

pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

print("Statistical summary added to PDF!")

Statistical summary added to PDF!


In [23]:
fig = plt.figure(figsize=(8.27, 11.69))
plt.axis("off")

plt.text(0.08, 0.94, "5. BOOKING TREND ANALYSIS",
         fontsize=17, fontweight="bold")

trend_text = """
Monthly Booking Analysis

Month          Total Rides       Revenue
--------------------------------------------
January          5,643           323,370.74
February         5,234           303,440.31
March            5,622           321,832.35
April            5,198           302,229.25
May              5,654           319,794.91
June             5,476           320,555.98
July             5,640           320,353.97
August           5,505           312,184.14
September        5,410           312,138.84
October            618            35,680.70

Key Findings

• May recorded the highest number of rides among
  the full months, with 5,654 bookings.
• October has only 618 rides because the available
  data covers October 1 to October 4.
• Monthly bookings remained relatively consistent
  from January to September.

Peak Booking Hour

• 7:00 PM recorded the highest activity, with
  2,161 rides.
• The lowest hourly count was at 6:00 PM,
  with 1,999 rides.

Business Insight

The evening period shows strong booking demand.
Driver availability can be planned according to
observed booking patterns.

Note: October represents a partial month.
"""

plt.text(0.08, 0.85, trend_text, fontsize=11.5,
         va="top", linespacing=1.6, family="monospace")

pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

print("Booking trend analysis added to PDF!")

Booking trend analysis added to PDF!


In [24]:
fig = plt.figure(figsize=(8.27, 11.69))
plt.axis("off")

plt.text(0.08, 0.94, "6. DRIVER PERFORMANCE ANALYSIS",
         fontsize=16, fontweight="bold")

driver_text = """
Driver performance was evaluated using actual ride
records grouped by driver ID.

Key Findings

• Jeremy Ramirez (Driver ID: 90) recorded the
  highest number of rides, with 201 rides.

• Lisa Rojas (Driver ID: 292) generated revenue
  of 12,480.69 from 197 rides.

• Alicia Moore (Driver ID: 128) completed 191 rides
  and had a rating of 4.71 among the top 10 drivers.

Analysis Approach

Actual ride counts were calculated from the rides
dataset using driver IDs.

The total_rides column in the drivers dataset was
not used for ranking because its values did not
consistently match the actual ride records.

Business Insights

• Tracking actual completed rides helps identify
  highly active drivers.
• Driver ratings can help evaluate service quality.
• Driver performance data can support better
  driver allocation and performance monitoring.
"""

plt.text(0.08, 0.85, driver_text, fontsize=12,
         va="top", linespacing=1.7)

pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

print("Driver performance analysis added to PDF!")

Driver performance analysis added to PDF!


In [25]:
fig = plt.figure(figsize=(8.27, 11.69))
plt.axis("off")

plt.text(0.08, 0.94, "7. PICKUP-LOCATION ANALYSIS",
         fontsize=17, fontweight="bold")

location_text = """
The pickup-location analysis identifies locations
with the highest number of recorded rides.

Top 10 Pickup Locations

1. South Michael       - 50 rides
2. Port Michael        - 48 rides
3. New Michael         - 46 rides
4. East Michael        - 44 rides
5. West Michael        - 42 rides
6. Lake Michael        - 41 rides
7. East Jennifer       - 37 rides
8. North Michael       - 36 rides
9. South Robert        - 34 rides
10. West Christopher   - 33 rides

Key Findings

• South Michael recorded the highest number of rides.
• Port Michael ranked second.
• The top locations can help identify areas with
  relatively higher booking activity.

Business Insight

Ride-sharing operations can use pickup-location
patterns to plan driver availability and improve
vehicle allocation.

Note: These are synthetic pickup-location names
provided in the dataset. They should not be treated
as verified real-world city rankings.
"""

plt.text(0.08, 0.85, location_text, fontsize=11.5,
         va="top", linespacing=1.7)

pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

print("Pickup-location analysis added to PDF!")

Pickup-location analysis added to PDF!


In [26]:
fig = plt.figure(figsize=(8.27, 11.69))
plt.axis("off")

plt.text(0.08, 0.94, "8. CUSTOMER RATING ANALYSIS",
         fontsize=17, fontweight="bold")

rating_text = """
Customer ratings were analyzed to understand
customer satisfaction with the ride-sharing service.

Rating Distribution

Rating          Number of Ratings
---------------------------------
1 Star              9,888
2 Stars             9,928
3 Stars            10,206
4 Stars             9,861
5 Stars            10,117

Key Findings

• 3-star ratings were the most common, with 10,206.
• 5-star ratings accounted for 10,117 records.
• 4-star ratings accounted for 9,861 records.
• The rating distribution is relatively balanced
  across all five rating levels.

Overall Average Customer Rating: 3.01 / 5

Business Insights

• Customer feedback can help identify areas for
  service improvement.
• Drivers can use rating feedback to improve
  customer experience.
• Regular rating monitoring can help track
  service quality over time.
"""

plt.text(0.08, 0.85, rating_text, fontsize=12,
         va="top", linespacing=1.7)

pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

print("Customer rating analysis added to PDF!")

Customer rating analysis added to PDF!


In [27]:
fig = plt.figure(figsize=(8.27, 11.69))
plt.axis("off")

plt.text(0.08, 0.94, "9. BUSINESS INSIGHTS & RECOMMENDATIONS",
         fontsize=15, fontweight="bold")

insights_text = """
KEY BUSINESS INSIGHTS

1. Booking Performance
The dataset contains 50,000 rides. Monthly booking
activity remained relatively consistent from January
to September 2024.

2. Revenue Performance
Total recorded revenue is 2,871,581.19, with an
average fare of 57.43 per ride.

3. Peak Demand
7:00 PM recorded the highest hourly booking activity.
Driver availability can be planned around demand.

4. Driver Performance
Jeremy Ramirez recorded the highest actual ride
count among drivers, with 201 rides.

5. Customer Satisfaction
The average customer rating is 3.01 out of 5.
Customer feedback can help identify service
improvement opportunities.

6. Pickup Locations
South Michael recorded the highest pickup count
among the listed locations, with 50 rides.

RECOMMENDATIONS

• Plan driver availability according to peak hours.
• Monitor driver performance and customer ratings.
• Review customer feedback to improve service quality.
• Use pickup-location trends for vehicle allocation.
• Monitor monthly bookings and revenue regularly.
• Maintain accurate and consistent operational data.

These recommendations are based on the available
dataset and its observed patterns.
"""

plt.text(0.08, 0.85, insights_text, fontsize=11.5,
         va="top", linespacing=1.65)

pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

print("Business insights and recommendations added!")

Business insights and recommendations added!


In [28]:
fig = plt.figure(figsize=(8.27, 11.69))
plt.axis("off")

plt.text(0.08, 0.94, "10. DATASET LIMITATIONS",
         fontsize=17, fontweight="bold")

limitations_text = """
During the analysis, the following limitations
were identified in the available datasets.

1. Cancellation Analysis
The rides dataset does not contain a ride status
or cancellation field. Therefore, cancellation rate
and cancellation count could not be calculated.

2. Payment Method Analysis
The dataset does not contain payment method details.
Therefore, payment distribution could not be analyzed.

3. City-wise Analysis
The pickup locations contain synthetic names.
They cannot be treated as verified real-world cities.

4. Partial October Data
October contains records only from October 1 to 4.
Therefore, it is not comparable with full months.

5. Driver Ride Count
The total_rides column in the drivers dataset does
not consistently match actual ride records.
Actual ride counts were calculated from the rides
dataset instead.

6. Customer Rating Relationships
Ratings were associated with rides using ride_id.
The ratings user_id field was not used for customer
linking because of inconsistencies.

CONCLUSION

The available datasets support analysis of booking
trends, revenue, trip distance, driver performance,
pickup locations and customer ratings.

Additional fields such as ride status, payment method
and verified city information would enable more
comprehensive operational analysis.
"""

plt.text(0.08, 0.85, limitations_text, fontsize=11.5,
         va="top", linespacing=1.7)

pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

print("Dataset limitations added to PDF!")

Dataset limitations added to PDF!


In [30]:
chart_files = [
    ("Monthly Booking Trend", "monthly_booking_trend.png"),
    ("Customer Rating Distribution", "customer_rating_distribution.png"),
    ("Fare and Distance Distribution", "fare_and_distance_distribution.png"),
    ("Distance vs Fare", "distance_vs_fare.png"),
    ("Top 10 Pickup Locations", "Top_10_Pickup_Locations.png"),
    ("Top 10 Drivers", "top_10_drivers.png"),
    ("Driver Rating Distribution", "driver_rating_distribution.png"),
    ("Correlation Heatmap", "correlation_heatmap.png"),
    ("Pair Plot", "pair_plot.png")
]

# Check whether all chart files exist
missing_charts = [
    filename for title, filename in chart_files
    if not os.path.exists(os.path.join(chart_folder, filename))
]

if missing_charts:
    print("Missing chart files:", missing_charts)

else:
    for title, filename in chart_files:
        image_path = os.path.join(chart_folder, filename)

        fig, ax = plt.subplots(figsize=(8.27, 11.69))
        chart_image = plt.imread(image_path)

        ax.imshow(chart_image)
        ax.axis("off")

        fig.suptitle(title, fontsize=17, fontweight="bold", y=0.95)

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

    print("All 9 existing charts added to PDF!")

All 9 existing charts added to PDF!


In [31]:
fig = plt.figure(figsize=(8.27, 11.69))
plt.axis("off")

plt.text(0.08, 0.94, "11. PROJECT CONCLUSION",
         fontsize=18, fontweight="bold")

conclusion_text = """
The Ride Sharing & Cab Booking Analysis project
provided an overview of ride-sharing operations
using SQL, Python, Pandas and Power BI.

The analysis covered:

• 50,000 ride records.
• Revenue and average fare analysis.
• Trip distance and ride duration.
• Monthly and hourly booking trends.
• Driver performance and ratings.
• Customer rating distribution.
• Pickup-location booking patterns.

The findings helped identify booking patterns,
revenue trends, driver performance and areas
where service quality could be improved.

The project also highlighted the importance of
data cleaning, accurate relationships and reliable
data for business decision-making.

Overall, this project demonstrates how data
analysis can convert raw operational data into
meaningful insights and recommendations.

"""

plt.text(0.08, 0.84, conclusion_text, fontsize=12,
         va="top", linespacing=1.8)

plt.text(0.5, 0.18, "END OF REPORT",
         fontsize=15, fontweight="bold", ha="center")

pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

print("Project conclusion added to PDF!")

Project conclusion added to PDF!


In [32]:
pdf.close()

print("PDF report saved and closed successfully!")

PDF report saved and closed successfully!


In [2]:
import os

report_path = r"C:\Users\adity\Theta A\DA Project\Ride_Sharing_Project_Report.pdf"

print("PDF Path:", report_path)

if os.path.exists(report_path):
    print("PDF Status: Found")
    print("File Size:", round(os.path.getsize(report_path) / 1024, 2), "KB")
else:
    print("PDF Status: Not Found")

PDF Path: C:\Users\adity\Theta A\DA Project\Ride_Sharing_Project_Report.pdf
PDF Status: Found
File Size: 1388.34 KB


In [3]:
import re

with open(report_path, "rb") as file:
    pdf_data = file.read()

page_count = len(re.findall(rb"/Type\s*/Page\b", pdf_data))

print("Total PDF Pages:", page_count)

Total PDF Pages: 29


In [4]:
import importlib.util

for library in ["PyPDF2", "fitz", "pdfplumber", "pypdf"]:
    print(library, ":", importlib.util.find_spec(library) is not None)

PyPDF2 : False
fitz : False
pdfplumber : False
pypdf : False
